# Deteksi Transaksi Kartu Kredit Penipuan dengan Random Forest
UTS Pembelajaran Mesin

Dataset: *Credit Card Fraud Detection* (ULB, Kaggle) - `creditcard.csv`. Unggah file ke Colab (ikon folder di kiri) atau unduh via Kaggle API.

In [ ]:
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, precision_recall_curve,
                             average_precision_score, roc_auc_score, ConfusionMatrixDisplay)

## 1. Memuat dan Eksplorasi Data

In [ ]:
df = pd.read_csv('creditcard.csv')
print(df.shape)
print(df.isnull().sum().sum(), 'nilai kosong;', df.duplicated().sum(), 'duplikat')
df['Class'].value_counts()

In [ ]:
ax = df['Class'].value_counts().plot(kind='bar', color=['steelblue','crimson'])
ax.set_xticklabels(['Normal (0)','Penipuan (1)'], rotation=0)
ax.set_title('Distribusi Kelas (sangat tidak seimbang)'); plt.show()
print((df['Class'].mean()*100).round(3), '% transaksi adalah penipuan')

## 2. Preprocessing
Fitur `V1-V28` sudah hasil PCA dan berskala sebanding. Random Forest tidak sensitif terhadap skala, jadi tidak perlu normalisasi. Duplikat dihapus. Data dibagi 80:20 dengan *stratified split* agar proporsi penipuan tetap sama.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
X = df.drop(columns='Class'); y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## 3. Pelatihan Model
Ketidakseimbangan kelas ditangani dengan `class_weight='balanced_subsample'`. Kedalaman pohon dibatasi agar file model tetap kecil untuk deployment.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=2,
                            class_weight='balanced_subsample', n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

## 4. Evaluasi
Akurasi menyesatkan pada data tidak seimbang (menebak semua 'normal' sudah ~99,8%). Metrik utama: *precision*, *recall*, F1, dan PR-AUC.

In [ ]:
proba = rf.predict_proba(X_test)[:,1]
pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, pred, target_names=['Normal','Penipuan'], digits=4))
print('ROC-AUC :', roc_auc_score(y_test, proba).round(4))
print('PR-AUC  :', average_precision_score(y_test, proba).round(4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['Normal','Penipuan']).plot(cmap='Blues')
plt.title('Confusion Matrix'); plt.show()

In [ ]:
p, r, t = precision_recall_curve(y_test, proba)
plt.plot(r, p); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve'); plt.show()

## 5. Pengaturan Threshold
Bisnis biasanya lebih mementingkan *recall* (penipuan tertangkap). Coba beberapa threshold dan bahas *trade-off*-nya di artikel.

In [ ]:
for th in [0.2,0.3,0.4,0.5,0.6]:
    pr = (proba>=th).astype(int)
    rep = classification_report(y_test, pr, output_dict=True, zero_division=0)['1']
    print(f"threshold {th}: precision={rep['precision']:.3f} recall={rep['recall']:.3f} f1={rep['f1-score']:.3f}")

## 6. Feature Importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
sns.barplot(x=imp.values, y=imp.index, color='steelblue'); plt.title('10 Fitur Terpenting'); plt.show()

## 7. Simpan Model dan Contoh Data untuk Aplikasi Web

In [ ]:
joblib.dump({'model': rf, 'columns': list(X.columns)}, 'model_fraud_rf.joblib', compress=3)
sample = pd.concat([X_test[y_test==1].sample(15, random_state=1), X_test[y_test==0].sample(35, random_state=1)])
sample.sample(frac=1, random_state=1).to_csv('sample_transaksi.csv', index=False)
import os; print(round(os.path.getsize('model_fraud_rf.joblib')/1e6,1), 'MB')
from google.colab import files
files.download('model_fraud_rf.joblib'); files.download('sample_transaksi.csv')